In [ ]:
 
import pandas as pd
import numpy as np
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

#Load encoded data 
X_train = pd.read_csv('/workspaces/Project_01/video_game_analysis/notebooks/FC110542_Weerakoon/Processed data/X_train_encoded.csv')
X_test = pd.read_csv('/workspaces/Project_01/video_game_analysis/notebooks/FC110542_Weerakoon/Processed data/X_test_encoded.csv')
y_train = pd.read_csv('/workspaces/Project_01/video_game_analysis/notebooks/FC110542_Weerakoon/Processed data/y_train_encoded.csv')
y_test = pd.read_csv('/workspaces/Project_01/video_game_analysis/notebooks/FC110542_Weerakoon/Processed data/y_test_encoded.csv')

print("✅ Encoded data loaded for tuning")
print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)

#Define parameter grid for tuning
param_grid = {
    'C': [0.1, 1, 10, 100],           # Regularization strength
    'epsilon': [0.01, 0.1, 0.2, 0.5], # Tolerance for error margin
    'gamma': ['scale', 'auto']        # Kernel coefficient
}

#Initialize base SVR model 
svr = SVR(kernel='rbf')

#Setup GridSearchCV 
grid_search = GridSearchCV(
    estimator=svr,
    param_grid=param_grid,
    scoring='r2',       # Evaluate based on R²
    cv=3,               # 3-fold cross-validation
    verbose=2,
    n_jobs=-1           # Use all available CPU cores
)

#Run Grid Search 
grid_search.fit(X_train, y_train.values.ravel())

print("\n Grid Search Complete")
print("Best Parameters:", grid_search.best_params_)
print("Best R² Score from CV:", grid_search.best_score_)

#Evaluate best model on test set 
best_svr = grid_search.best_estimator_
y_pred = best_svr.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print(f"\n📊 Final Model Performance on Test Set:")
print(f"RMSE: {rmse:.4f}")
print(f"R² Score: {r2:.4f}")

#Visualize predictions
plt.figure(figsize=(6,6))
plt.scatter(y_test, y_pred, alpha=0.4)
plt.xlabel("Actual Global_Sales_Log")
plt.ylabel("Predicted Global_Sales_Log")
plt.title("Tuned SVR Predictions vs Actuals")
plt.plot(
    [y_test.min()[0], y_test.max()[0]],
    [y_test.min()[0], y_test.max()[0]],
    color='red', linestyle='--'
)
plt.show()
